# Chapter 10: Design a Notification System
**From:** *System Design Interview*

---

## TL;DR

- A notification system supports three channels: **mobile push** (APNs/FCM), **SMS** (Twilio/Nexmo), and **email** (SendGrid/Mailchimp).
- The improved design decouples ingestion from delivery using **per-channel message queues** with **worker pools**, enabling independent scaling and fault isolation.
- **Reliability** is achieved through a notification log database + retry mechanism; **deduplication** via event ID checks reduces duplicate sends.
- Key additions: **notification templates** for consistency, **rate limiting** to prevent user overload, **authentication** via appKey/appSecret, and **analytics tracking** for open/click rates.
- The system is **soft real-time** -- notifications arrive quickly but slight delays under load are acceptable.

---
## Requirements

| Requirement | Value |
|---|---|
| Notification types | Push, SMS, Email |
| Real-time? | Soft real-time (slight delay OK) |
| Devices | iOS, Android, Desktop |
| Triggers | Client apps + server-side scheduled |
| Opt-out | Yes |
| Volume | 10M push + 1M SMS + 5M email / day |

---
## Back-of-the-Envelope Estimation

In [ ]:
# Notification System Estimation
push_per_day = 10_000_000
sms_per_day = 1_000_000
email_per_day = 5_000_000
total_per_day = push_per_day + sms_per_day + email_per_day

seconds_per_day = 24 * 3600
avg_qps = total_per_day / seconds_per_day
peak_qps = avg_qps * 3  # assume 3x peak

# Assume avg notification payload
avg_payload_bytes = 1024  # 1 KB per notification
daily_data_gb = total_per_day * avg_payload_bytes / (1024**3)

print(f"Total notifications/day:  {total_per_day:>12,}")
print(f"  Push:                   {push_per_day:>12,}")
print(f"  SMS:                    {sms_per_day:>12,}")
print(f"  Email:                  {email_per_day:>12,}")
print(f"Average QPS:              {avg_qps:>12,.0f}")
print(f"Peak QPS (3x):            {peak_qps:>12,.0f}")
print(f"Daily data volume:        {daily_data_gb:>12,.1f} GB")

---
## High-Level Design

```mermaid
graph LR
    S1[Service 1] --> NS[Notification Servers]
    S2[Service 2] --> NS
    SN[Service N] --> NS
    NS --> Cache[(Cache)]
    NS --> DB[(Database)]
    NS --> Q1[iOS PN Queue]
    NS --> Q2[Android PN Queue]
    NS --> Q3[SMS Queue]
    NS --> Q4[Email Queue]
    Q1 --> W1[Workers] --> APNS[APNs]
    Q2 --> W2[Workers] --> FCM[FCM]
    Q3 --> W3[Workers] --> Twilio[Twilio]
    Q4 --> W4[Workers] --> SendGrid[SendGrid]
    APNS --> iOS[iOS Device]
    FCM --> Android[Android Device]
    Twilio --> Phone[Phone]
    SendGrid --> Inbox[Email Inbox]
```

### Component Responsibilities

| Component | Role |
|---|---|
| **Notification Servers** | API gateway with auth, validation, rate limiting; routes to queues |
| **Message Queues** | Per-channel buffers decouple ingestion from delivery; fault isolation |
| **Workers** | Pull from queues, format payloads, call third-party services |
| **Third-party Services** | APNs (iOS), FCM (Android), Twilio (SMS), SendGrid (Email) |
| **Cache** | User info, device tokens, notification templates |
| **Database** | User settings, notification log, contact info |

---
## Deep Dive: Reliability

### Data Loss Prevention
- Every notification is persisted to a **notification log database** before sending
- If delivery fails, the notification can be retried from the log

### Deduplication
- When a notification event arrives, check its **event ID** against previously seen events
- If already seen, discard it (prevents duplicates from distributed retries)
- **Exactly-once delivery is impossible** in distributed systems -- this minimizes duplicates

### Retry Mechanism
```mermaid
graph LR
    Worker --> ThirdParty[Third-party Service]
    ThirdParty -->|Failure| RetryQueue[Retry Queue]
    RetryQueue --> Worker
    ThirdParty -->|Repeated Failure| Alert[Developer Alert]
```

---
## Deep Dive: Additional Components

| Component | Purpose |
|---|---|
| **Notification Templates** | Preformatted layouts with parameterized fields; ensures consistency |
| **Notification Settings** | Per-user, per-channel opt-in/opt-out stored in DB; checked before every send |
| **Rate Limiting** | Frequency cap per user to prevent notification fatigue and opt-out |
| **Security** | appKey/appSecret pair for API authentication; only verified clients send |
| **Event Tracking** | Open rate, click rate, engagement metrics fed to analytics service |
| **Queue Monitoring** | Track queued notification count; scale workers if backlog grows |

---
## Trade-offs

| Decision | Option A | Option B | Recommendation |
|---|---|---|---|
| Architecture | Single server | Distributed with queues | Distributed -- avoids SPOF and bottleneck |
| Queue strategy | Single shared queue | Per-channel queues | Per-channel -- fault isolation per type |
| Delivery guarantee | At-most-once | At-least-once + dedup | At-least-once -- never lose notifications |
| Template vs. custom | Build each notification | Use templates | Templates -- consistency + speed at scale |
| Third-party services | Build in-house | Use commercial providers | Commercial -- better deliverability + analytics |

---
## Key Takeaways

1. **Message queues are the backbone.** Per-channel queues decouple notification servers from delivery workers, enabling independent scaling and fault isolation.

2. **Never lose a notification.** Persist to a log database before sending, and implement retries with exponential backoff for failed deliveries.

3. **Respect the user.** Check opt-in settings before every send; enforce rate limits to prevent notification fatigue that drives users to disable all notifications.

4. **Exactly-once delivery is a myth.** In distributed systems, design for at-least-once delivery with deduplication to minimize (but not eliminate) duplicates.

5. **Templates scale content creation.** With millions of notifications daily, parameterized templates ensure consistency and reduce errors.

6. **Monitor queue depth.** If queued notifications grow faster than workers process them, auto-scale workers to maintain delivery latency.

---
## See Also

- [[notification-types]] -- Push, SMS, and Email delivery channels and their providers
- [[message-queue-decoupling]] -- Per-channel message queues for fault isolation
- [[reliability-and-retry]] -- Notification log persistence and retry mechanisms
- [[deduplication]] -- Event ID-based deduplication for distributed delivery
- [[rate-limiting-and-opt-in]] -- Frequency capping and user notification preferences
- [[notification-templates]] -- Parameterized notification layouts for consistency